In [4]:
import os
import re
import anndata as ad
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from tqdm import tqdm

def compute_spatial_similarity(
    adata_msi, adata_rna, moran_msi_csv, moran_rna_csv,
    moran_threshold=0.1, top_n_per_msi=10
):
    """Compute MSI-RNA spatial similarity for one MSI and one RNA sample."""

    moran_msi = pd.read_csv(moran_msi_csv)
    moran_rna = pd.read_csv(moran_rna_csv)

    high_msi = moran_msi[moran_msi['moran_statistic'] > moran_threshold]['feature'].tolist()
    high_rna = moran_rna[moran_rna['moran_statistic'] > moran_threshold]['feature'].tolist()

    print(f"  • {len(high_msi)} MSI and {len(high_rna)} RNA features passed Moran threshold")

    # Align coordinates (rounded)
    msi_coords = adata_msi.obs[['x', 'y']].round(2)
    rna_coords = adata_rna.obs[['x', 'y']].round(2)
    merged = pd.merge(msi_coords, rna_coords, on=['x', 'y'], how='inner')

    print(f"  • Matched {len(merged)} pixels")

    if len(merged) < 100:
        print("  ⚠️ Too few overlapping pixels — skipping")
        return pd.DataFrame()

    # Mask data
    msi_mask = adata_msi.obs[['x', 'y']].round(2).apply(tuple, axis=1).isin([tuple(xy) for xy in merged[['x', 'y']].values])
    rna_mask = adata_rna.obs[['x', 'y']].round(2).apply(tuple, axis=1).isin([tuple(xy) for xy in merged[['x', 'y']].values])

    msi_data = adata_msi[msi_mask, high_msi].X
    rna_data = adata_rna[rna_mask, high_rna].X

    if hasattr(msi_data, "toarray"): msi_data = msi_data.toarray()
    if hasattr(rna_data, "toarray"): rna_data = rna_data.toarray()

    # DataFrames for correlation
    msi_df = pd.DataFrame(msi_data, columns=high_msi)
    rna_df = pd.DataFrame(rna_data, columns=high_rna)

    results = []
    for m in tqdm(high_msi, desc="    Correlating MSI vs RNA features"):
        m_vec = msi_df[m]
        for g in high_rna:
            corr, _ = spearmanr(m_vec, rna_df[g])
            if not np.isnan(corr):
                results.append((m, g, corr))

    if not results:
        return pd.DataFrame()

    corr_df = pd.DataFrame(results, columns=['m_z', 'gene', 'spearman_corr'])
    corr_df['abs_corr'] = corr_df['spearman_corr'].abs()

    # Rank and keep top N per MSI
    ranked = (
        corr_df.sort_values('abs_corr', ascending=False)
        .groupby('m_z')
        .head(top_n_per_msi)
        .reset_index(drop=True)
    )

    return ranked


def batch_process_similarity(
    msi_dir, rna_dir, moran_dir, output_dir,
    moran_threshold=0.1, top_n_per_msi=10
):
    """Process all MSI and RNA samples automatically."""

    os.makedirs(output_dir, exist_ok=True)
    msi_files = sorted([f for f in os.listdir(msi_dir) if f.endswith('.h5ad')])
    rna_files = sorted([f for f in os.listdir(rna_dir) if f.endswith('.h5ad')])

    for msi_file in msi_files:
        # Load MSI sample
        msi_path = os.path.join(msi_dir, msi_file)
        adata_msi = ad.read_h5ad(msi_path)
        sample_id = re.sub(r'\.h5ad$', '', msi_file)
        moran_msi_csv = os.path.join(moran_dir, f"{sample_id}_moran.csv")

        if not os.path.exists(moran_msi_csv):
            print(f"⚠️ Missing Moran CSV for {msi_file}, skipping")
            continue

        for rna_file in rna_files:
            rna_path = os.path.join(rna_dir, rna_file)
            adata_rna = ad.read_h5ad(rna_path)
            rna_id = re.sub(r'\.h5ad$', '', rna_file)
            moran_rna_csv = os.path.join(moran_dir, f"{rna_id}_moran.csv")

            if not os.path.exists(moran_rna_csv):
                print(f"⚠️ Missing Moran CSV for {rna_file}, skipping")
                continue

            print(f"\n🔹 Comparing {sample_id} (MSI) ↔ {rna_id} (RNA)")

            ranked = compute_spatial_similarity(
                adata_msi, adata_rna,
                moran_msi_csv, moran_rna_csv,
                moran_threshold=moran_threshold,
                top_n_per_msi=top_n_per_msi
            )

            if not ranked.empty:
                output_name = f"{sample_id}_vs_{rna_id}_spatial_similarity.csv"
                ranked.to_csv(os.path.join(output_dir, output_name), index=False)
                print(f"✅ Saved: {output_name}")
            else:
                print(f"⚠️ No significant correlations for {sample_id} vs {rna_id}")


# 🧠 Example usage
if __name__ == "__main__":
    batch_process_similarity(
        msi_dir="/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common",
        rna_dir="/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes",
        moran_dir="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_results",
        output_dir="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_similarity",
        moran_threshold=0.15,
        top_n_per_msi=20
    )


🔹 Comparing msi_aad_1_20 (MSI) ↔ rna_aad_1_20 (RNA)


KeyError: 'feature'

In [ ]:
"""
Improved MSI-RNA Spatial Similarity Analysis
Computes spatial correlation between MSI features and RNA genes
"""

import os
import re
import anndata as ad
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from tqdm import tqdm
from pathlib import Path


def load_moran_csv(csv_path, data_type="MSI"):
    """
    Load Moran statistics CSV with flexible column detection.
    
    Args:
        csv_path: Path to Moran CSV file
        data_type: "MSI" or "RNA" for informative error messages
        
    Returns:
        DataFrame with columns: feature, moran_statistic
    """
    try:
        # Read CSV with first column as index (unnamed column contains features)
        df = pd.read_csv(csv_path, index_col=0)
        
        # The index contains the features (m/z values or gene names)
        features = df.index.astype(str).tolist()
        
        # Detect Moran statistic column
        moran_col = None
        for col in df.columns:
            if 'moran' in col.lower() and 'stat' in col.lower():
                moran_col = col
                break
        
        if moran_col is None:
            raise ValueError(f"No 'moran_statistic' column found in {csv_path}")
        
        # Create standardized DataFrame
        result = pd.DataFrame({
            'feature': features,
            'moran_statistic': df[moran_col].astype(float).values
        })
        
        print(f"  ✓ Loaded {len(result)} {data_type} features from Moran CSV")
        return result
        
    except Exception as e:
        print(f"  ✗ Error loading {csv_path}: {e}")
        return None


def align_spatial_coordinates(adata_msi, adata_rna, decimal_places=0):
    """
    Align spatial coordinates between MSI and RNA datasets.
    
    Args:
        adata_msi: MSI AnnData object
        adata_rna: RNA AnnData object
        decimal_places: Number of decimal places for rounding coordinates
        
    Returns:
        Tuple of (msi_mask, rna_mask, n_matched)
    """
    # Try different coordinate column names (prefer x, y over x_um, y_um for matching)
    coord_options = [
        ['x', 'y'],
        ['X', 'Y'],
        ['x_um', 'y_um'],
        ['x_coord', 'y_coord']
    ]
    
    msi_coords = None
    rna_coords = None
    used_coords = None
    
    # Find coordinate columns in both datasets (use same coordinate system)
    for coord_cols in coord_options:
        if all(col in adata_msi.obs.columns for col in coord_cols) and \
           all(col in adata_rna.obs.columns for col in coord_cols):
            msi_coords = adata_msi.obs[coord_cols].copy()
            rna_coords = adata_rna.obs[coord_cols].copy()
            used_coords = coord_cols
            break
    
    if msi_coords is None or rna_coords is None:
        raise ValueError("Could not find matching spatial coordinates in both datasets")
    
    print(f"  • Using coordinates: {used_coords}")
    
    # Round coordinates for matching
    if decimal_places > 0:
        msi_coords = msi_coords.round(decimal_places)
        rna_coords = rna_coords.round(decimal_places)
    else:
        # For integer coordinates, convert to int
        msi_coords = msi_coords.astype(int)
        rna_coords = rna_coords.astype(int)
    
    # Standardize column names for merging
    msi_coords_std = msi_coords.copy()
    rna_coords_std = rna_coords.copy()
    msi_coords_std.columns = ['x', 'y']
    rna_coords_std.columns = ['x', 'y']
    
    # Find overlapping coordinates
    merged = pd.merge(msi_coords_std, rna_coords_std, on=['x', 'y'], how='inner')
    n_matched = len(merged)
    
    if n_matched == 0:
        print("  ⚠️ No overlapping coordinates found")
        return None, None, 0
    
    # Create masks for boolean indexing
    merged_tuples = set(merged.apply(tuple, axis=1))
    msi_mask = msi_coords_std.apply(tuple, axis=1).isin(merged_tuples)
    rna_mask = rna_coords_std.apply(tuple, axis=1).isin(merged_tuples)
    
    return msi_mask, rna_mask, n_matched


def compute_spatial_similarity(
    adata_msi, 
    adata_rna, 
    moran_msi_csv, 
    moran_rna_csv,
    moran_threshold=0.1, 
    top_n_per_msi=10,
    min_overlap_pixels=100
):
    """
    Compute MSI-RNA spatial similarity for one MSI and one RNA sample.
    
    Args:
        adata_msi: MSI AnnData object
        adata_rna: RNA AnnData object
        moran_msi_csv: Path to MSI Moran statistics CSV
        moran_rna_csv: Path to RNA Moran statistics CSV
        moran_threshold: Minimum Moran's I value for filtering features
        top_n_per_msi: Number of top correlated genes to keep per MSI feature
        min_overlap_pixels: Minimum number of overlapping spatial pixels required
        
    Returns:
        DataFrame with columns: m_z, gene, spearman_corr, abs_corr
    """
    
    # Load Moran statistics
    moran_msi = load_moran_csv(moran_msi_csv, "MSI")
    moran_rna = load_moran_csv(moran_rna_csv, "RNA")
    
    if moran_msi is None or moran_rna is None:
        return pd.DataFrame()
    
    # Filter by Moran threshold
    high_msi = moran_msi[moran_msi['moran_statistic'] > moran_threshold]['feature'].tolist()
    high_rna = moran_rna[moran_rna['moran_statistic'] > moran_threshold]['feature'].tolist()
    
    print(f"  • {len(high_msi)} MSI and {len(high_rna)} RNA features passed Moran threshold (>{moran_threshold})")
    
    if len(high_msi) == 0 or len(high_rna) == 0:
        print("  ⚠️ No features passed threshold — skipping")
        return pd.DataFrame()
    
    # Align spatial coordinates
    msi_mask, rna_mask, n_matched = align_spatial_coordinates(adata_msi, adata_rna)
    
    if n_matched < min_overlap_pixels:
        print(f"  ⚠️ Only {n_matched} overlapping pixels (need ≥{min_overlap_pixels}) — skipping")
        return pd.DataFrame()
    
    print(f"  • Matched {n_matched} spatial pixels")
    
    # Filter features that exist in the data
    msi_var_names = set(adata_msi.var_names.astype(str))
    rna_var_names = set(adata_rna.var_names.astype(str))
    
    high_msi_valid = [f for f in high_msi if f in msi_var_names]
    high_rna_valid = [f for f in high_rna if f in rna_var_names]
    
    if len(high_msi_valid) < len(high_msi):
        print(f"  ⚠️ {len(high_msi) - len(high_msi_valid)} MSI features not found in data")
    if len(high_rna_valid) < len(high_rna):
        print(f"  ⚠️ {len(high_rna) - len(high_rna_valid)} RNA features not found in data")
    
    if len(high_msi_valid) == 0 or len(high_rna_valid) == 0:
        print("  ⚠️ No valid features found — skipping")
        return pd.DataFrame()
    
    # Extract data for matched pixels
    try:
        msi_data = adata_msi[msi_mask, high_msi_valid].X
        rna_data = adata_rna[rna_mask, high_rna_valid].X
    except Exception as e:
        print(f"  ✗ Error extracting data: {e}")
        return pd.DataFrame()
    
    # Convert sparse to dense if needed
    if hasattr(msi_data, "toarray"):
        msi_data = msi_data.toarray()
    if hasattr(rna_data, "toarray"):
        rna_data = rna_data.toarray()
    
    # Create DataFrames
    msi_df = pd.DataFrame(msi_data, columns=high_msi_valid)
    rna_df = pd.DataFrame(rna_data, columns=high_rna_valid)
    
    # Compute correlations
    results = []
    print(f"  • Computing {len(high_msi_valid)} × {len(high_rna_valid)} = {len(high_msi_valid) * len(high_rna_valid)} correlations")
    
    for m in tqdm(high_msi_valid, desc="    Correlating features", leave=False):
        m_vec = msi_df[m].values
        
        # Skip if all zeros or constant
        if np.std(m_vec) == 0:
            continue
            
        for g in high_rna_valid:
            g_vec = rna_df[g].values
            
            # Skip if all zeros or constant
            if np.std(g_vec) == 0:
                continue
            
            try:
                corr, pval = spearmanr(m_vec, g_vec)
                if not np.isnan(corr):
                    results.append({
                        'm_z': m,
                        'gene': g,
                        'spearman_corr': corr,
                        'p_value': pval
                    })
            except Exception as e:
                continue
    
    if not results:
        print("  ⚠️ No valid correlations computed")
        return pd.DataFrame()
    
    # Create results DataFrame
    corr_df = pd.DataFrame(results)
    corr_df['abs_corr'] = corr_df['spearman_corr'].abs()
    
    print(f"  • Computed {len(corr_df)} valid correlations")
    
    # Rank and keep top N per MSI feature
    ranked = (
        corr_df.sort_values('abs_corr', ascending=False)
        .groupby('m_z')
        .head(top_n_per_msi)
        .reset_index(drop=True)
    )
    
    return ranked


def batch_process_similarity(
    msi_dir, 
    rna_dir, 
    moran_dir, 
    output_dir,
    moran_threshold=0.1, 
    top_n_per_msi=10,
    min_overlap_pixels=100,
    msi_pattern="*.h5ad",
    rna_pattern="*.h5ad"
):
    """
    Process all MSI and RNA samples automatically.
    
    Args:
        msi_dir: Directory containing MSI h5ad files
        rna_dir: Directory containing RNA h5ad files
        moran_dir: Directory containing Moran statistics CSV files
        output_dir: Directory to save results
        moran_threshold: Minimum Moran's I for feature selection
        top_n_per_msi: Number of top genes to keep per MSI feature
        min_overlap_pixels: Minimum overlapping spatial pixels required
        msi_pattern: File pattern for MSI files (e.g., "*.h5ad" or "halfbrain_*.h5ad")
        rna_pattern: File pattern for RNA files
    """
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all files
    msi_files = sorted(Path(msi_dir).glob(msi_pattern))
    rna_files = sorted(Path(rna_dir).glob(rna_pattern))
    
    print("=" * 80)
    print("MSI-RNA SPATIAL SIMILARITY ANALYSIS")
    print("=" * 80)
    print(f"MSI samples: {len(msi_files)}")
    print(f"RNA samples: {len(rna_files)}")
    print(f"Moran threshold: {moran_threshold}")
    print(f"Top N per MSI: {top_n_per_msi}")
    print(f"Min overlap pixels: {min_overlap_pixels}")
    print("=" * 80 + "\n")
    
    if len(msi_files) == 0:
        print(f"⚠️ No MSI files found in {msi_dir}")
        return
    
    if len(rna_files) == 0:
        print(f"⚠️ No RNA files found in {rna_dir}")
        return
    
    # Process each combination
    total_comparisons = len(msi_files) * len(rna_files)
    successful = 0
    failed = 0
    
    for i, msi_file in enumerate(msi_files, 1):
        # Load MSI sample
        try:
            adata_msi = ad.read_h5ad(msi_file)
            sample_id = msi_file.stem.replace('.h5ad', '')
            moran_msi_csv = Path(moran_dir) / f"{sample_id}_moran.csv"
            
            if not moran_msi_csv.exists():
                print(f"⚠️ Missing Moran CSV for {msi_file.name}, skipping")
                failed += len(rna_files)
                continue
                
        except Exception as e:
            print(f"✗ Error loading {msi_file.name}: {e}")
            failed += len(rna_files)
            continue
        
        for j, rna_file in enumerate(rna_files, 1):
            comparison_num = (i - 1) * len(rna_files) + j
            
            try:
                # Load RNA sample
                adata_rna = ad.read_h5ad(rna_file)
                rna_id = rna_file.stem.replace('.h5ad', '')
                moran_rna_csv = Path(moran_dir) / f"{rna_id}_moran.csv"
                
                if not moran_rna_csv.exists():
                    print(f"⚠️ Missing Moran CSV for {rna_file.name}, skipping")
                    failed += 1
                    continue
                
                print(f"\n[{comparison_num}/{total_comparisons}] 🔹 {sample_id} (MSI) ↔ {rna_id} (RNA)")
                
                # Compute similarity
                ranked = compute_spatial_similarity(
                    adata_msi, 
                    adata_rna,
                    moran_msi_csv, 
                    moran_rna_csv,
                    moran_threshold=moran_threshold,
                    top_n_per_msi=top_n_per_msi,
                    min_overlap_pixels=min_overlap_pixels
                )
                
                if not ranked.empty:
                    # Save results
                    output_name = f"{sample_id}_vs_{rna_id}_spatial_similarity.csv"
                    output_path = Path(output_dir) / output_name
                    ranked.to_csv(output_path, index=False)
                    print(f"✅ Saved: {output_name} ({len(ranked)} correlations)")
                    successful += 1
                else:
                    print(f"⚠️ No significant correlations for {sample_id} vs {rna_id}")
                    failed += 1
                    
            except Exception as e:
                print(f"✗ Error processing {sample_id} vs {rna_id}: {e}")
                failed += 1
                continue
    
    print("\n" + "=" * 80)
    print("PROCESSING COMPLETE")
    print("=" * 80)
    print(f"✅ Successful: {successful}/{total_comparisons}")
    print(f"⚠️ Failed: {failed}/{total_comparisons}")
    print("=" * 80)


# Example usage
if __name__ == "__main__":
    batch_process_similarity(
        msi_dir="/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common",
        rna_dir="/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes",
        moran_dir="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_results",
        output_dir="/home/ajarrah/PhD_Thesis/chapter_4/results/spatial_similarity",
        moran_threshold=0.15,
        top_n_per_msi=20,
        min_overlap_pixels=100
    )

MSI-RNA SPATIAL SIMILARITY ANALYSIS
MSI samples: 16
RNA samples: 16
Moran threshold: 0.15
Top N per MSI: 20
Min overlap pixels: 100


[1/256] 🔹 msi_aad_1_20 (MSI) ↔ rna_aad_1_20 (RNA)
  ✓ Loaded 528 MSI features from Moran CSV
  ✓ Loaded 7928 RNA features from Moran CSV
  • 528 MSI and 7692 RNA features passed Moran threshold (>0.15)
  • Using coordinates: ['x', 'y']
  • Matched 34830 spatial pixels
  • Computing 528 × 7692 = 4061376 correlations


    Correlating features:   1%|          | 4/528 [02:17<5:01:04, 34.47s/it]

In [7]:
adata = ad.read_h5ad('/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/lipids_common/msi_aad_1_20.h5ad')

In [8]:
adata

AnnData object with n_obs × n_vars = 34866 × 528
    obs: 'sample', 'y', 'x', 'x_um', 'y_um'
    obsm: 'spatial'

In [9]:
adata = ad.read_h5ad('/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes/rna_aad_1_20.h5ad')

In [10]:
adata

AnnData object with n_obs × n_vars = 58289 × 7928
    obs: 'sample', 'y', 'x', 'x_um', 'y_um'
    obsm: 'spatial'